In [1]:
import pandas as pd
import re
from numpy import mean, nan
import subprocess, requests, tarfile, os
from zipfile import ZipFile
from pybarrnap import Barrnap
from pybarrnap.utils import load_example_fasta_file

# Read in the data
df = pd.read_csv("eukaryotes_ncbi_temperatures.csv")

columns = ["#Organism Name", "Organism Groups", "Assembly", "Temperature (°C)"]
df = df[columns]
df = df.dropna(subset="Temperature (°C)")
df = df.loc[df['Organism Groups'].str.contains("Fung")]
df.drop_duplicates(subset="Assembly")

# Extract species root names
df['species_root_name'] = df['#Organism Name'].apply(lambda item: " ".join(item.split(" ")[:2]))
df['species_root_name'] = df['species_root_name'].apply(lambda item: re.sub(r"[\[\];'\",\(\).;\-]", "", item))

# Identify duplicates based on species root names
df['duplicate'] = df.duplicated(subset="species_root_name", keep=False)

# Remove entries with " cf. " in the organism name
df = df.loc[~df['#Organism Name'].str.contains(" cf. ")]
df = df.reset_index(drop=True)
df.head()

,#Organism Name,Organism Groups,Assembly,Temperature (°C),species_root_name,duplicate
0,Schizosaccharomyces pombe,Eukaryota;Fungi;Ascomycetes,GCA_000002945.2,25.0,Schizosaccharomyces pombe,False
1,Aspergillus nidulans FGSC A4,Eukaryota;Fungi;Ascomycetes,GCA_000149205.2,26.0,Aspergillus nidulans,False
2,Aspergillus fumigatus Af293,Eukaryota;Fungi;Ascomycetes,GCA_000002655.1,28.0,Aspergillus fumigatus,False
3,Neurospora crassa OR74A,Eukaryota;Fungi;Ascomycetes,GCA_000182925.2,25.0,Neurospora crassa,False
4,Candida albicans SC5314,Eukaryota;Fungi;Ascomycetes,GCA_000182965.3,25.0,Candida albicans,False


In [7]:
# get fasta files
# unzip fasta
# barrnap fasta file for quality.
# get tRNA & save it in df
# scrape web for it????
is_quality = []
quality_scores = []
quality_means = []
trna_fasta = []

for row in range(1):
    assembly = df.at[row, 'Assembly']
    url = f"https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/{assembly}/download?include_annotation_type=GENOME_FASTA"
    res = requests.get(url)  # this returns a zip folder

    # Define the filename for the downloaded zip file
    zip_filename = f"{assembly}.zip"
    # Save the zip file
    with open(zip_filename, 'wb') as f:
        for chunk in res.iter_content(chunk_size=8192):
            f.write(chunk)
    extract_dir = f"{assembly}_dir"
    if not f"{extract_dir}":
        os.mkdir(extract_dir)

    with ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    
    # Name path of the extracted folder
    fasta_path = f"{extract_dir}/ncbi_dataset/data/{assembly}"

    fasta_file = os.listdir(fasta_path)[0] # => []
    # Run pybarrnap rRNA prediction
    barrnap = Barrnap(
        fasta_path + "/" + fasta_file,
        evalue=1e-6,
        lencutoff=0.8,
        reject=0.25,
        threads=1,
        kingdom="euk",
        accurate=False,
        quiet=False,
    )
    result = barrnap.run()
    # Get rRNA GFF text and print
    # print("\n========== Print rRNA GFF ==========")
    gff_text = result.get_gff_text()
    gff_text1 = [lines for lines in gff_text.split("\n")]
    gff_nums = [
        float(item.split("\t")[5]) for item in gff_text1[1:] if len(item.split('\t'))>6
        ]
    gff_mean = mean(gff_nums)
    print(gff_text)
    print(f"gff_mean: {gff_mean}")

    # Get rRNA features and print
    print("\n========== Print rRNA features ==========")
    genes_set = set()
    needed_genes = {"5S", "5_8S", "18S", "28S"}
    for rec in result.seq_records: # seq_records is a list
        print(rec)
        for feature in rec.features: # features on rec is a list
            qualifiers = feature.qualifiers #feature.id, feature.type, feature.location, feature.qualifiers)
            if "gene" in qualifiers:
                for g in qualifiers["gene"]:
                    genes_set.add(g)

    return_value = None # Quality Metric
    if all([any([needed in g for g in genes_set]) for needed in needed_genes]):
        return_value = True
        quality_nums = gff_nums
        quality_mean = gff_mean
    else:
        return_value = False
        quality_nums = nan
        quality_mean = nan

    fasta_full_path = fasta_path + "/" + fasta_file
    output_name = f"{fasta_file}_trna.fasta"
    !tRNAscan-SE {fasta_full_path} --fasta {output_name}

    is_quality.append(return_value)
    quality_scores.append(quality_nums)
    quality_means.append(quality_mean)
    trna_fasta.append(output_name)
            # qaul_genes = []
            # all three types had to be identified
            # must = '5S', '5_8S', '18S', '28S'

    # Do stuff here with the Fasta file!
    # ✔️ Pybarrnap
    # ✔️Create a quality metric & save to df
    # Save quality metric to df
    # Find tRNA from the fasta
    # Save tRNA to df

    %pwd
    %ls
    %rm -rf {extract_dir}
    %rm {zip_filename}

    print("\n")
    print(f"{'='*10}")
    print("\n")
    %ls
    # !ncbi-genome-download --section genbank --assembly-accessions {assembly} --formats fasta fungi

2026-08-12 16:28:42 | INFO | Run pybarrnap v0.5.2
2026-08-12 16:28:42 | INFO | Operating System: linux
2026-08-12 16:28:42 | INFO | Python Version: v3.14.6
2026-08-12 16:28:42 | INFO | Check Dependencies: pyhmmer v0.12.1 is installed
2026-08-12 16:28:42 | INFO | Check Dependencies: biopython v1.88 is installed
2026-08-12 16:28:42 | INFO | Set Option: evalue=1e-06
2026-08-12 16:28:42 | INFO | Set Option: lencutoff=0.8
2026-08-12 16:28:42 | INFO | Set Option: reject=0.25
2026-08-12 16:28:42 | INFO | Set Option: threads=1
2026-08-12 16:28:42 | INFO | Set Option: kingdom='euk'
2026-08-12 16:28:42 | INFO | Set Option: accurate=False
2026-08-12 16:28:42 | INFO | Number of Target Sequence = 4
2026-08-12 16:28:42 | INFO | Seq1. name='CU329670.1', length=5,579,133, description='CU329670.1 Schizosaccharomyces pombe strain 972h- genome assembly, chromosome: I'
2026-08-12 16:28:42 | INFO | Seq2. name='CU329671.1', length=4,539,804, description='CU329671.1 Schizosaccharomyces pombe strain 972h- gen

##gff-version 3
CU329670.1	pybarrnap:0.5.2	rRNA	149660	149774	7.8e-22	+	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	456529	456643	4.7e-22	+	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	912232	912346	1.9e-21	-	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	1563477	1563591	2.1e-21	+	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	2976821	2976935	7.8e-22	+	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	3165398	3165512	7.8e-22	+	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	3547902	3548016	7.9e-20	-	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	3634373	3634487	7.8e-22	+	.	Name=5S_rRNA;product=5S ribosomal RNA;Dbxref=RFAM:RF00001
CU329670.1	pybarrnap:0.5.2	rRNA	3830382	383049